In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_33.pickle

In [ ]:
%%RecordEvent
%%time
### cell 34 ###

def grab_subset_of_data_46(df, question):
    # select and rename all columns containing the question text, then drop rows with all NaNs
    cols = [c for c in df.columns if question in c]
    new_cols = [c.rsplit('-', 1)[-1].lstrip() for c in cols]
    sub = df.loc[:, cols].rename(columns=dict(zip(cols, new_cols)))
    return sub.dropna(how='all', subset=new_cols)


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
        question, include_2017=False):
    # prepare list of (year, dataframe) and extract subsets in one pass
    sources = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017:
        sources.append(('2017', responses_df_2017))
    dfs = []
    for year, src in sources:
        sub = grab_subset_of_data_46(src, question)
        sub['year'] = year
        dfs.append(sub)
    # concatenate in ascending-year order
    dfs = sorted(dfs, key=lambda d: d['year'].iat[0])
    df_combined = pd.concat(dfs, ignore_index=True)
    df_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_counts


def convert_df_of_counts_to_percentages_46(df, df_counts):
    # vectorized conversion: divide by total respondents per year
    totals = df['year'].value_counts()
    df_perc = df_counts.copy()
    count_cols = df_perc.columns.difference(['year'])
    df_perc[count_cols] = df_perc[count_cols].div(df_perc['year'].map(totals)).mul(100)
    return df_perc


# rename 2018 columns once (matches question_of_interest in grab_subset)
question_of_interest_old = 'What machine learning frameworks have you used in the past 5 years?'
question_of_interest_new_cell46 = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018_cell10 = responses_df_2018_cell10.rename(
    columns={question_of_interest_old: question_of_interest_new_cell46}
)

question_of_interest_cell46 = question_of_interest_new_cell46
ml_df_combined, ml_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
        question_of_interest_cell46
    )

# define which raw columns to coalesce into one for each option
combine_mapping = {
    'TensorFlow ': ['TensorFlow', 'TensorFlow '],
    'Keras ': ['Keras', 'Keras '],
    'PyTorch ': ['PyTorch', 'PyTorch '],
    'Scikit-learn ': ['Scikit—learn ', 'Learn', 'learn '],
}

# build a new DataFrame with one column per option
ml = ml_df_combined.copy()
for new_col, old_cols in combine_mapping.items():
    ml[new_col] = (
        ml[old_cols]
        .notna()
        .any(axis=1)
        .map({True: new_col.strip(), False: np.nan})
    )
ml_df_combined_cell46 = ml.drop(columns=[c for cols in combine_mapping.values() for c in cols])

# recalculate counts and percentages
ml_df_combined_counts_2 = ml_df_combined_cell46.groupby('year').count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_46(
    ml_df_combined_cell46, ml_df_combined_counts_2
)

# select and melt for plotting
cols_to_keep = ['year'] + list(combine_mapping.keys()) + ['None', 'Other']
ml_df_combined_percentages_cell46 = ml_df_combined_percentages.loc[:, cols_to_keep]

df_cell46 = ml_df_combined_percentages_cell46.melt(
    id_vars=['year'],
    value_vars=list(combine_mapping.keys()) + ['None', 'Other']
)
df_cell46 = df_cell46.sort_values(by=['year', 'value'])
df_cell46 = df_cell46.rename(columns={'variable': ''})
df_cell46.info()


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_34_try_0.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_34_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[34], f)


In [ ]:
opt_output = Out.get(4)